In [0]:
# Daily gold snapshot: guest satisfaction per property
# Source: samples.wanderbricks.reviews (non-deleted) + samples.wanderbricks.properties
# Target: {catalog}.{schema}.gold_property_guest_satisfaction_daily (one row per property per snapshot_date)
from datetime import date

dbutils.widgets.text("catalog", "genie_code_workshop")
dbutils.widgets.text("schema", "asml_demo")
dbutils.widgets.text("snapshot_date", "")  # empty = today

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
snapshot_date = dbutils.widgets.get("snapshot_date") or date.today().isoformat()
date.fromisoformat(snapshot_date)  # validate format (YYYY-MM-DD) before using it in replaceWhere

target = f"{catalog}.{schema}.gold_property_guest_satisfaction_daily"
print(target, snapshot_date)

In [0]:
from pyspark.sql import functions as F

# Soft-deleted reviews (is_deleted = true) are excluded from every metric
reviews = (
    spark.table("samples.wanderbricks.reviews")
    .filter(~F.col("is_deleted") & F.col("rating").isNotNull())
)
properties = spark.table("samples.wanderbricks.properties").select(
    "property_id", F.col("title").alias("property_title"), "property_type", "destination_id"
)

agg = reviews.groupBy("property_id").agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("rating"), 2).alias("avg_rating"),
    F.sum(F.when(F.col("rating") >= 4, 1).otherwise(0)).alias("satisfied_review_count"),
    F.sum(F.when(F.col("rating") <= 2, 1).otherwise(0)).alias("dissatisfied_review_count"),
    F.max("created_at").alias("last_review_at"),
)

df = (
    agg.join(properties, "property_id", "inner")
    .withColumn("pct_satisfied", F.round(100 * F.col("satisfied_review_count") / F.col("review_count"), 2))
    .withColumn("pct_dissatisfied", F.round(100 * F.col("dissatisfied_review_count") / F.col("review_count"), 2))
    .withColumn("net_satisfaction_score", F.round(F.col("pct_satisfied") - F.col("pct_dissatisfied"), 2))
    .withColumn(
        "satisfaction_band",
        F.when(F.col("avg_rating") >= 4.5, "HIGH").when(F.col("avg_rating") >= 3.5, "MEDIUM").otherwise("LOW"),
    )
    .withColumn("snapshot_date", F.lit(snapshot_date).cast("date"))
    .withColumn("audit_timestamp", F.current_timestamp())
    .withColumn("source_system", F.lit("samples.wanderbricks"))
    # identifiers -> dimensions -> measures -> derived -> audit columns LAST
    .select(
        "snapshot_date", "property_id",
        "property_title", "property_type", "destination_id",
        "review_count", "satisfied_review_count", "dissatisfied_review_count", "last_review_at",
        "avg_rating", "pct_satisfied", "pct_dissatisfied", "net_satisfaction_score", "satisfaction_band",
        "audit_timestamp", "source_system",
    )
)

if df.limit(1).count() == 0:
    raise ValueError(f"No rows produced for {target}")

In [0]:
# Re-running for the same snapshot_date replaces only that day's rows (no duplicates)
writer = df.write.format("delta")
if spark.catalog.tableExists(target):
    writer.mode("overwrite").option("replaceWhere", f"snapshot_date = '{snapshot_date}'").saveAsTable(target)
else:
    writer.mode("overwrite").saveAsTable(target)

print(f"Wrote {spark.table(target).filter(F.col('snapshot_date') == snapshot_date).count()} rows to {target} for {snapshot_date}")

In [0]:
# DDL per table-governance standard; re-applied on every run
spark.sql(f"""COMMENT ON TABLE {target} IS
  'Business aggregation: guest satisfaction (review count, average rating, satisfied and dissatisfied share, net satisfaction score) by property and snapshot_date. Soft-deleted reviews excluded.'""")

spark.sql(f"""ALTER TABLE {target} SET TBLPROPERTIES (
  'quality' = 'gold',
  'domain' = 'guest_experience',
  'contains_pii' = 'false',
  'delta.enableChangeDataFeed' = 'true')""")

column_comments = {
    "snapshot_date": "Snapshot date of the daily job run. Part of the primary key (snapshot_date, property_id)",
    "property_id": "Part of the primary key. References samples.wanderbricks.properties.property_id",
    "property_title": "Title of the property listing, from samples.wanderbricks.properties.title",
    "property_type": "Type of property (house, apartment, ...), from samples.wanderbricks.properties",
    "destination_id": "References samples.wanderbricks.destinations.destination_id",
    "review_count": "COUNT of non-deleted reviews with a rating for the property",
    "satisfied_review_count": "COUNT of non-deleted reviews with rating >= 4 stars",
    "dissatisfied_review_count": "COUNT of non-deleted reviews with rating <= 2 stars",
    "last_review_at": "MAX(created_at) of non-deleted reviews for the property",
    "avg_rating": "Calculated as AVG(rating) over non-deleted reviews, rounded to 2 decimals. Units: stars 1-5",
    "pct_satisfied": "Calculated as 100 * satisfied_review_count / review_count. Units: percent 0-100",
    "pct_dissatisfied": "Calculated as 100 * dissatisfied_review_count / review_count. Units: percent 0-100",
    "net_satisfaction_score": "Calculated as pct_satisfied - pct_dissatisfied. Units: points, range -100 to 100",
    "satisfaction_band": "Derived from avg_rating: HIGH >= 4.5, MEDIUM >= 3.5, else LOW",
    "audit_timestamp": "Job execution timestamp",
    "source_system": "Upstream source system identifier",
}
for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {target} ALTER COLUMN {col} COMMENT '{comment}'")

# Note: governed tag policies in this account reject two of the standard tags:
#   'pii' allows only 'salary' -> left off (TBLPROPERTIES contains_pii = 'false' records it)
#   'domain' has no allowed values -> left off (TBLPROPERTIES domain = 'guest_experience' records it)
spark.sql(f"""ALTER TABLE {target} SET TAGS (
  'quality' = 'gold', 'data_classification' = 'internal', 'sla' = 'daily')""")

In [0]:
# Copy of asml-workshop/skills/job-standards/scripts/check_table_standards.py (metadata reads only)
import re

LAYER_PREFIXES = ("bronze_", "silver_", "gold_")
REQUIRED_PROPS = ("quality", "domain")
AUDIT_COLUMNS = ["audit_timestamp", "source_system"]
NAME_PATTERN = re.compile(r"^[a-z][a-z0-9_]*$")

def check_table_standards(full_name: str) -> list:
    failures = []
    table = full_name.split(".")[2]
    if not NAME_PATTERN.match(table):
        failures.append(f"Table name '{table}' is not lowercase_snake_case")
    if not table.startswith(LAYER_PREFIXES):
        failures.append(f"Table name '{table}' has no layer prefix {LAYER_PREFIXES}")
    if not spark.sql(f"DESCRIBE DETAIL {full_name}").first()["description"]:
        failures.append("Missing table COMMENT")
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()}
    failures += [f"Missing TBLPROPERTY '{p}'" for p in REQUIRED_PROPS if p not in props]
    if props.get("contains_pii") == "true" and not props.get("pii_columns"):
        failures.append("contains_pii=true but 'pii_columns' is not set")
    fields = spark.table(full_name).schema.fields
    if [f.name for f in fields][-2:] != AUDIT_COLUMNS:
        failures.append(f"Last two columns must be {AUDIT_COLUMNS}")
    undocumented = [f.name for f in fields if not f.metadata.get("comment")]
    if undocumented:
        failures.append(f"Columns without COMMENT: {undocumented}")
    print(f"[{'PASS' if not failures else f'FAIL ({len(failures)})'}] {full_name}")
    for f in failures:
        print(f"  - {f}")
    return failures

failures = check_table_standards(target)
if failures:
    raise AssertionError(f"{target} does not meet table standards: {failures}")

# Primary key must be unique for this snapshot
dupes = spark.table(target).filter(F.col("snapshot_date") == snapshot_date).groupBy("property_id").count().filter("count > 1").count()
if dupes:
    raise AssertionError(f"{dupes} duplicate property_id rows for {snapshot_date}")